In [0]:
# Catálogo principal
spark.sql("CREATE CATALOG IF NOT EXISTS fintech_finpay")
# Schemas del catálogo
schemas = ["default", "bronze", "silver", "gold", "observability"]
# Creación de schemas
for schema in schemas:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS fintech_finpay.{schema}")
    print(f"Schema creado exitosamente: fintech_finpay.{schema}")
# Validar schemas creados
display(spark.sql("SHOW SCHEMAS IN fintech_finpay"))

Schema creado exitosamente: fintech_finpay.default
Schema creado exitosamente: fintech_finpay.bronze
Schema creado exitosamente: fintech_finpay.silver
Schema creado exitosamente: fintech_finpay.gold
Schema creado exitosamente: fintech_finpay.observability


databaseName
bronze
default
gold
information_schema
observability
silver


In [0]:
# Creación de Volume de landing
spark.sql(""" 
          CREATE VOLUME IF NOT EXISTS fintech_finpay.default.vol_landing
        """)
print("El Volume fue creado existosamente: fintech_finpay.default.vol_landing")
# Se define la ruta de Volumes y nombres de subdirectorios
base_path = "/Volumes/fintech_finpay/default/vol_landing"
subdirs = ["transactions", "merchants", "users", "metadata"]
# Creación de subdirectorios
for subdir in subdirs:
    path = f"{base_path}/{subdir}"
    dbutils.fs.mkdirs(path)
    print(f"Subdirectorio creado exitosamente: {path}")
display(dbutils.fs.ls(base_path))

El Volume fue creado existosamente: fintech_finpay.default.vol_landing
Subdirectorio creado exitosamente: /Volumes/fintech_finpay/default/vol_landing/transactions
Subdirectorio creado exitosamente: /Volumes/fintech_finpay/default/vol_landing/merchants
Subdirectorio creado exitosamente: /Volumes/fintech_finpay/default/vol_landing/users
Subdirectorio creado exitosamente: /Volumes/fintech_finpay/default/vol_landing/metadata


path,name,size,modificationTime
dbfs:/Volumes/fintech_finpay/default/vol_landing/merchants/,merchants/,0,1779056649327
dbfs:/Volumes/fintech_finpay/default/vol_landing/metadata/,metadata/,0,1779056649327
dbfs:/Volumes/fintech_finpay/default/vol_landing/transactions/,transactions/,0,1779056649327
dbfs:/Volumes/fintech_finpay/default/vol_landing/users/,users/,0,1779056649327


In [0]:
# ROL INGENIERIA
# Creación de permiso de catalogo
spark.sql("GRANT USE CATALOG ON CATALOG fintech_finpay TO ingenieria")
# Creación de permisos sobre schemas para el rol de Ingenieria
for schema in ["default", "bronze", "silver", "gold", "observability"]:
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA fintech_finpay.{schema} TO ingenieria")
    spark.sql(f"GRANT CREATE TABLE ON SCHEMA fintech_finpay.{schema} TO ingenieria")
    spark.sql(f"GRANT MODIFY ON SCHEMA fintech_finpay.{schema} TO ingenieria")
    spark.sql(f"GRANT SELECT ON SCHEMA fintech_finpay.{schema} TO ingenieria")
print("Se dieron los permisos asignados para el rol de ingenieria.")

# ROL RIESGO
# Creación de permiso de catalogo
spark.sql("GRANT USE CATALOG ON CATALOG fintech_finpay TO riesgo")
# Creación de permisos sobre schemas para el rol de riesgo
for schema in ["silver","gold"]:
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA fintech_finpay.{schema} TO riesgo")
    spark.sql(f"GRANT SELECT ON SCHEMA fintech_finpay.{schema} TO riesgo")
print("Se dieron los permisos asignados para el rol de riesgo.")

# ROL AUDITORIA
# Creación de permiso de catalogo
spark.sql("GRANT USE CATALOG ON CATALOG fintech_finpay TO auditoria")
# Creación de permisos sobre schemas para el rol de auditoria
for schema in ["gold","observability"]:
    spark.sql(f"GRANT USE SCHEMA ON SCHEMA fintech_finpay.{schema} TO auditoria")
    spark.sql(f"GRANT SELECT ON SCHEMA fintech_finpay.{schema} TO auditoria")
print("Se dieron los permisos asignados para el rol de auditoria.")

print("Todos los roles han sido configurados correctamente")

Se dieron los permisos asignados para el rol de ingenieria.
Se dieron los permisos asignados para el rol de riesgo.
Se dieron los permisos asignados para el rol de auditoria.
Todos los roles han sido configurados correctamente


In [0]:
# Crear tabla silver.users
spark.sql("""
    CREATE TABLE IF NOT EXISTS fintech_finpay.silver.users (
        user_id           STRING,
        full_name         STRING,
        document_id       STRING,
        email             STRING,
        phone             STRING,
        country           STRING,
        segment           STRING,
        registration_date DATE
    )
""")
print("Tabla silver.users creada existosamente")

# Función de masking
spark.sql("""
    CREATE OR REPLACE FUNCTION fintech_finpay.silver.mask_pii(value STRING)
    RETURNS STRING
    RETURN CASE
        WHEN IS_ACCOUNT_GROUP_MEMBER('ingenieria') THEN value
        ELSE '***MASKED***'
    END
""")
print("Función de masking creada exitosamente")

# Masking a campos PII
for field in ["full_name", "document_id", "email", "phone"]:
    spark.sql(f"""
        ALTER TABLE fintech_finpay.silver.users
        ALTER COLUMN {field}
        SET MASK fintech_finpay.silver.mask_pii
    """)
    print(f"Column masking aplicado existosamente: {field}")

# Función de row-level security: Solo ingenieria puede acceder a toda la data
spark.sql("""
    CREATE OR REPLACE FUNCTION fintech_finpay.silver.row_filter_users(country STRING)
    RETURNS BOOLEAN
    RETURN IS_ACCOUNT_GROUP_MEMBER('ingenieria')
""")
print("Función de row-level security creada")

# Aplicar RLS
spark.sql("""
    ALTER TABLE fintech_finpay.silver.users
    SET ROW FILTER fintech_finpay.silver.row_filter_users ON (country)
""")
print("Row-level security aplicado en silver.users")

Tabla silver.users creada existosamente
Función de masking creada exitosamente
Column masking aplicado existosamente: full_name
Column masking aplicado existosamente: document_id
Column masking aplicado existosamente: email
Column masking aplicado existosamente: phone
✓ Función de row-level security creada
✓ Row-level security aplicado en silver.users


In [0]:
spark.sql("SHOW SCHEMAS IN fintech_finpay").show()
dbutils.fs.ls("/Volumes/fintech_finpay/default/vol_landing")
spark.sql("DESCRIBE TABLE fintech_finpay.silver.users").show()

+------------------+
|      databaseName|
+------------------+
|            bronze|
|           default|
|              gold|
|information_schema|
|     observability|
|            silver|
+------------------+

+-----------------+---------+-------+
|         col_name|data_type|comment|
+-----------------+---------+-------+
|          user_id|   string|   NULL|
|        full_name|   string|   NULL|
|      document_id|   string|   NULL|
|            email|   string|   NULL|
|            phone|   string|   NULL|
|          country|   string|   NULL|
|          segment|   string|   NULL|
|registration_date|     date|   NULL|
+-----------------+---------+-------+



In [ ]:
-- ============================================================
-- BLOQUE 6: Vistas por schema para organización por capas
-- Las tablas DLT viven en default — las vistas exponen
-- la misma data organizadas en bronze, silver y gold
-- ============================================================

-- Bronze
CREATE OR REPLACE VIEW fintech_finpay.bronze.raw_transactions 
AS SELECT * FROM fintech_finpay.default.raw_transactions;

CREATE OR REPLACE VIEW fintech_finpay.bronze.raw_merchants 
AS SELECT * FROM fintech_finpay.default.raw_merchants;

CREATE OR REPLACE VIEW fintech_finpay.bronze.raw_users 
AS SELECT * FROM fintech_finpay.default.raw_users;

-- Silver
CREATE OR REPLACE VIEW fintech_finpay.silver.transactions 
AS SELECT * FROM fintech_finpay.default.transactions;

CREATE OR REPLACE VIEW fintech_finpay.silver.merchants 
AS SELECT * FROM fintech_finpay.default.merchants;

CREATE OR REPLACE VIEW fintech_finpay.silver.quarantine 
AS SELECT * FROM fintech_finpay.default.quarantine;

-- Gold
CREATE OR REPLACE VIEW fintech_finpay.gold.risk_kpis 
AS SELECT * FROM fintech_finpay.default.risk_kpis;

CREATE OR REPLACE VIEW fintech_finpay.gold.anomalies 
AS SELECT * FROM fintech_finpay.default.anomalies;

CREATE OR REPLACE VIEW fintech_finpay.gold.channel_summary 
AS SELECT * FROM fintech_finpay.default.channel_summary;